# HDFS Viewer Notebook

This notebook allows you to inspect the contents of the Hadoop Distributed File System (HDFS) and preview data using PySpark.

## 1. List Files in HDFS
We start by listing all files and directories in the HDFS root to understand the structure.

In [10]:
!hdfs dfs -ls -R /

drwxr-xr-x   - ubuntu supergroup          0 2026-03-05 23:36 /nyc_taxi_data
drwxr-xr-x   - ubuntu supergroup          0 2026-03-05 23:37 /nyc_taxi_data/strong_10GB
-rw-r--r--   3 ubuntu supergroup  491076642 2026-03-05 23:36 /nyc_taxi_data/strong_10GB/m1.parquet
-rw-r--r--   3 ubuntu supergroup  461632254 2026-03-05 23:36 /nyc_taxi_data/strong_10GB/m2.parquet
-rw-r--r--   3 ubuntu supergroup  501783767 2026-03-05 23:37 /nyc_taxi_data/strong_10GB/m3.parquet
-rw-r--r--   3 ubuntu supergroup  487219586 2026-03-05 23:37 /nyc_taxi_data/strong_10GB/m4.parquet
-rw-r--r--   3 ubuntu supergroup  517386544 2026-03-05 23:37 /nyc_taxi_data/strong_10GB/m5.parquet
-rw-r--r--   3 ubuntu supergroup  490819474 2026-03-05 23:37 /nyc_taxi_data/strong_10GB/m6.parquet
drwxr-xr-x   - ubuntu supergroup          0 2026-03-05 23:36 /nyc_taxi_data/weak_3GB
-rw-r--r--   3 ubuntu supergroup  491076642 2026-03-05 23:36 /nyc_taxi_data/weak_3GB/m1.parquet
-rw-r--r--   3 ubuntu supergroup  461632254 2026-03-05 23:36 

## 2. Initialize Spark Session
To read the data files (which appear to be in Parquet format based on the directory names and contents), we'll use PySpark.

In [11]:
import os
import sys

# Set JAVA_HOME
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

# Set SPARK_HOME environment variable to the location of the installed pyspark package
# This can help if Spark cannot find its own directories
# Note: The path below assumes pyspark is installed in the venv
venv_site_packages = os.path.join(os.environ.get("VIRTUAL_ENV", ""), "lib", f"python{sys.version_info.major}.{sys.version_info.minor}", "site-packages")
spark_home = os.path.join(venv_site_packages, "pyspark")

if os.path.exists(spark_home):
    os.environ["SPARK_HOME"] = spark_home
    # Add PySpark properly to sys.path if needed
    sys.path.insert(0, os.path.join(spark_home, "python"))
    sys.path.insert(0, os.path.join(spark_home, "python", "lib", "py4j-0.10.9.7-src.zip"))

print(f"JAVA_HOME set to: {os.environ.get('JAVA_HOME')}")
print(f"SPARK_HOME set to: {os.environ.get('SPARK_HOME')}")

JAVA_HOME set to: /usr/lib/jvm/java-17-openjdk-amd64
SPARK_HOME set to: /home/ubuntu/1TD169_62015_DE1/venv/lib/python3.10/site-packages/pyspark


In [12]:
from pyspark.sql import SparkSession

# Initialize Spark Session
spark = SparkSession.builder \
    .appName("HDFS Viewer") \
    .master("local[*]") \
    .config("fs.defaultFS", "hdfs://g11-master:9000") \
    .getOrCreate()

print("Spark Session created successfully.")

Spark Session created successfully.


## 3. Read and Preview Data
We will load the `weak_3GB` dataset from `/nyc_taxi_data/weak_3GB` and display its schema and first few rows.

In [13]:
# Define the path to the dataset
#dataset_path = "/nyc_taxi_data/weak_3GB"
dataset_path = "hdfs://g11-master:9000/nyc_taxi_data/weak_3GB"

# Read the Parquet data
df = spark.read.parquet(dataset_path)

# Print the schema
print("Schema:")
df.printSchema()

# Show the first 5 rows
print("First 5 rows:")
df.show(5)

Schema:
root
 |-- hvfhs_license_num: string (nullable = true)
 |-- dispatching_base_num: string (nullable = true)
 |-- originating_base_num: string (nullable = true)
 |-- request_datetime: timestamp_ntz (nullable = true)
 |-- on_scene_datetime: timestamp_ntz (nullable = true)
 |-- pickup_datetime: timestamp_ntz (nullable = true)
 |-- dropoff_datetime: timestamp_ntz (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- trip_miles: double (nullable = true)
 |-- trip_time: long (nullable = true)
 |-- base_passenger_fare: double (nullable = true)
 |-- tolls: double (nullable = true)
 |-- bcf: double (nullable = true)
 |-- sales_tax: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)
 |-- tips: double (nullable = true)
 |-- driver_pay: double (nullable = true)
 |-- shared_request_flag: string (nullable = true)
 |-- shared_match_flag: string (nullable = true)
 |-- a

+-----------------+--------------------+--------------------+-------------------+-------------------+-------------------+-------------------+------------+------------+----------+---------+-------------------+-----+----+---------+--------------------+-----------+----+----------+-------------------+-----------------+------------------+----------------+--------------+------------------+
|hvfhs_license_num|dispatching_base_num|originating_base_num|   request_datetime|  on_scene_datetime|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|trip_miles|trip_time|base_passenger_fare|tolls| bcf|sales_tax|congestion_surcharge|airport_fee|tips|driver_pay|shared_request_flag|shared_match_flag|access_a_ride_flag|wav_request_flag|wav_match_flag|cbd_congestion_fee|
+-----------------+--------------------+--------------------+-------------------+-------------------+-------------------+-------------------+------------+------------+----------+---------+-------------------+-----+----+-------

## 4. Basic Statistics
Let's count the total number of records in the dataset to verify access.

In [14]:
# Count rows
record_count = df.count()
print(f"Total number of records in '{dataset_path}': {record_count}")

Total number of records in 'hdfs://g11-master:9000/nyc_taxi_data/weak_3GB': 39745127


## 5. Stop Spark Session
It is good practice to stop the Spark session when you are finished to release resources.

In [15]:
# Stop the Spark session
spark.stop()
print("Spark Session stopped.")

Spark Session stopped.
